# Lock benchmark design

This notebook defines and freezes the benchmark specification for the instruction-distraction robustness project.

It captures:
- the 5 benchmark tasks
- the 8 prompt conditions used at evaluation time (clean + 7 distraction families)
- the 2 prompt regimes
- the authority structure of the bounded regime
- task output formats and scoring assumptions
- dataset-size accounting
- canonicalization utilities for exact-match style evaluation
- an exportable benchmark specification JSON

In [1]:
from dataclasses import dataclass, field, asdict
from typing import List, Dict, Any
import json
from pathlib import Path
import sys

In [2]:
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

PROJECT_ROOT

WindowsPath('C:/code/testing/LLMs_Robustness_Under_Distractions')

## Specification dataclasses

In [3]:
@dataclass
class TaskSpec:
    name: str
    description: str
    output_format: Dict[str, Any]
    scoring_rule: str
    constraints: List[str] = field(default_factory=list)


@dataclass
class DistractionFamilySpec:
    name: str
    description: str
    notes: List[str] = field(default_factory=list)


@dataclass
class PromptRegimeSpec:
    name: str
    description: str
    properties: List[str] = field(default_factory=list)


@dataclass
class DatasetSizeSpec:
    base_examples_per_task: int
    num_tasks: int
    num_distraction_families: int
    num_regimes: int
    total_base_examples: int
    total_clean_prompt_instances: int
    total_distracted_prompt_instances: int
    total_prompt_instances: int


@dataclass
class BenchmarkSpec:
    benchmark_name: str
    benchmark_goal: str
    tasks: List[TaskSpec]
    distraction_families: List[DistractionFamilySpec]
    prompt_regimes: List[PromptRegimeSpec]
    bounded_regime_note: str
    dataset_size: DatasetSizeSpec
    scoring_notes: List[str] = field(default_factory=list)

## Prompt regimes

We now define the regimes in a way that matches the redesigned prompt layer.

Important design change:
- bounded prompts keep visible tagged sections like `<TASK>` and `<INPUT>`, but vary in opener, layout, and surrounding phrasing
- unbounded prompts are not simply tagless bounded prompts; they are rendered as distinct naturalistic surfaces

In [4]:
prompt_regimes = [
    PromptRegimeSpec(
        name="bounded",
        description=(
            "Controlled condition with explicit tagged sections. "
            "The prompt preserves visible task/input boundaries while still varying "
            "in opener, layout, and surrounding phrasing."
        ),
        properties=[
            "Uses visible tagged sections such as <TASK> and <INPUT>.",
            "Includes subtle sampled openers rather than one rigid authority sentence.",
            "Allows multiple bounded layouts, including variation in section order and labels.",
            "Is intended as a controlled condition, not as a universal model of real prompting.",
        ],
    ),
    PromptRegimeSpec(
        name="unbounded",
        description=(
            "Naturalistic condition without formal tagged task/input boundaries. "
            "Prompts are rendered as message-like, memo-like, paste-like, chat-like, "
            "email-like, or workflow-like user-facing surfaces."
        ),
        properties=[
            "No formal tagged authority boundary is used.",
            "Surface form is intentionally more realistic and varied.",
            "The regime is not derived by simply stripping tags from bounded prompts.",
            "The goal is to approximate messier real-world prompting conditions.",
        ],
    ),
]

## Bounded regime note

This is a conceptual design note for the thesis write-up, not a literal prompt string.

In [5]:
bounded_regime_note = (
    "In the bounded regime, the prompt visibly separates the task and the accompanying material "
    "using tagged sections. The tags provide a controlled structural boundary, but the prompt is "
    "not otherwise made artificially trivial: opener wording, section ordering, and surrounding "
    "phrasing may vary across instances."
)

print(bounded_regime_note)

In the bounded regime, the prompt visibly separates the task and the accompanying material using tagged sections. The tags provide a controlled structural boundary, but the prompt is not otherwise made artificially trivial: opener wording, section ordering, and surrounding phrasing may vary across instances.


## Task specifications

These are benchmark-level task definitions, not per-instance templates.
They should match the redesigned base-data pipeline.

In [6]:
tasks = [
    TaskSpec(
        name="single_label_classification",
        description=(
            "Given an input text, assign exactly one label from a fixed closed label set."
        ),
        output_format={
            "type": "json_object",
            "schema": {"label": "string"},
            "example": {"label": "positive"},
        },
        scoring_rule="exact_match_on_canonical_json",
        constraints=[
            "Output must be valid JSON.",
            "Output must contain exactly one key: 'label'.",
            "The label must belong to the instance-specific allowed label set.",
            "No extra keys are allowed.",
        ],
    ),
    TaskSpec(
        name="multi_label_classification",
        description=(
            "Given an input text, assign all applicable labels from a fixed closed label set."
        ),
        output_format={
            "type": "json_object",
            "schema": {"labels": ["string"]},
            "example": {"labels": ["health", "tech"]},
        },
        scoring_rule="exact_match_on_canonical_json_after_label_sorting",
        constraints=[
            "Output must be valid JSON.",
            "Output must contain exactly one key: 'labels'.",
            "The value of 'labels' must be a list of strings.",
            "Labels must belong to the instance-specific allowed label set.",
            "Labels should be canonicalized to unique sorted order for scoring.",
        ],
    ),
    TaskSpec(
        name="information_extraction",
        description=(
            "Extract a fixed set of fields from the text and return them in structured form."
        ),
        output_format={
            "type": "json_object",
            "schema": {
                "person": "string",
                "date": "string",
                "location": "string",
            },
            "example": {
                "person": "Alice Smith",
                "date": "2025-04-30",
                "location": "Rome",
            },
        },
        scoring_rule="exact_match_on_canonical_json",
        constraints=[
            "Output must be valid JSON.",
            "Output must contain exactly the required schema keys.",
            "All extracted values must be strings.",
            "No extra keys are allowed.",
        ],
    ),
    TaskSpec(
        name="rule_based_transformation",
        description=(
            "Apply a deterministic text transformation and return the transformed text."
        ),
        output_format={
            "type": "json_object",
            "schema": {"text": "string"},
            "example": {"text": "alice visited rome."},
        },
        scoring_rule="exact_match_on_canonical_json",
        constraints=[
            "Output must be valid JSON.",
            "Output must contain exactly one key: 'text'.",
            "The transformed text must match the deterministic gold output exactly.",
        ],
    ),
    TaskSpec(
        name="extractive_qa",
        description=(
            "Answer a question using an exact span from the passage."
        ),
        output_format={
            "type": "json_object",
            "schema": {"answer": "string"},
            "example": {"answer": "Rome"},
        },
        scoring_rule="exact_match_on_canonical_json",
        constraints=[
            "Output must be valid JSON.",
            "Output must contain exactly one key: 'answer'.",
            "The answer must match the gold span exactly.",
            "At the dataset level, QA examples should be constructed so the gold span is unique in the passage.",
        ],
    ),
]

len(tasks)

5

## Distraction families

At evaluation time, we have:
- 1 clean condition
- 7 distraction families

The clean condition is not itself a distraction family, but it is part of the total prompt-instance accounting.

In [7]:
distraction_families = [
    DistractionFamilySpec(
        name="irrelevant_prefix",
        description="Insert a moderate irrelevant block before the core prompt.",
        notes=[
            "Uses the expanded short-noise inventory.",
            "Should not use self-revealing background disclaimers.",
        ],
    ),
    DistractionFamilySpec(
        name="irrelevant_suffix",
        description="Insert a moderate irrelevant block after the core prompt.",
        notes=[
            "Uses the expanded short-noise inventory.",
            "Should preserve task content unchanged.",
        ],
    ),
    DistractionFamilySpec(
        name="instruction_in_the_middle",
        description="Bury the core prompt between two irrelevant blocks.",
        notes=[
            "This stresses retrieval of the relevant instruction region.",
            "Should use two distinct noise blocks where possible.",
        ],
    ),
    DistractionFamilySpec(
        name="conflicting_instruction",
        description="Add a realistic conflicting instruction outside the intended task.",
        notes=[
            "Includes realistic subtypes such as conflicting task, format override, output-mode override, summary request, and late override note.",
            "Should not rely only on a single absurd token such as 'pineapple'.",
        ],
    ),
    DistractionFamilySpec(
        name="negation_distraction",
        description="Add a misleading negation or softened reversal of the intended task.",
        notes=[
            "Includes direct negation, paraphrastic negation, partial-scope negation, format negation, and softened counterinstructions.",
            "Should avoid repeatedly negating the exact instruction wording in a trivial way.",
        ],
    ),
    DistractionFamilySpec(
        name="style_distraction",
        description="Add stylistic pressure that conflicts with literal terse task execution.",
        notes=[
            "Should use longer, stronger style guidance rather than very short caricature prompts only.",
            "May include formal, dramatic, legalistic, helpdesk-like, poetic, promotional, or other styles.",
        ],
    ),
    DistractionFamilySpec(
        name="length_stress",
        description="Insert a substantially longer irrelevant block near the prompt.",
        notes=[
            "Should include varied long-noise genres such as report fragments, documentation, code-plus-comments, procedural manuals, policy text, or email-thread fragments.",
            "Should not contain self-labeling statements that reveal it is irrelevant.",
        ],
    ),
]

len(distraction_families)

7

## Dataset-size accounting

Prompt-instance accounting includes:
- clean bounded
- clean unbounded
- distracted bounded
- distracted unbounded

In [8]:
BASE_EXAMPLES_PER_TASK = 50
NUM_TASKS = len(tasks)
NUM_DISTRACTION_FAMILIES = len(distraction_families)
NUM_REGIMES = len(prompt_regimes)

TOTAL_BASE_EXAMPLES = BASE_EXAMPLES_PER_TASK * NUM_TASKS
TOTAL_CLEAN_PROMPT_INSTANCES = TOTAL_BASE_EXAMPLES * NUM_REGIMES
TOTAL_DISTRACTED_PROMPT_INSTANCES = TOTAL_BASE_EXAMPLES * NUM_DISTRACTION_FAMILIES * NUM_REGIMES
TOTAL_PROMPT_INSTANCES = TOTAL_CLEAN_PROMPT_INSTANCES + TOTAL_DISTRACTED_PROMPT_INSTANCES

dataset_size = DatasetSizeSpec(
    base_examples_per_task=BASE_EXAMPLES_PER_TASK,
    num_tasks=NUM_TASKS,
    num_distraction_families=NUM_DISTRACTION_FAMILIES,
    num_regimes=NUM_REGIMES,
    total_base_examples=TOTAL_BASE_EXAMPLES,
    total_clean_prompt_instances=TOTAL_CLEAN_PROMPT_INSTANCES,
    total_distracted_prompt_instances=TOTAL_DISTRACTED_PROMPT_INSTANCES,
    total_prompt_instances=TOTAL_PROMPT_INSTANCES,
)

print(asdict(dataset_size))

{'base_examples_per_task': 50, 'num_tasks': 5, 'num_distraction_families': 7, 'num_regimes': 2, 'total_base_examples': 250, 'total_clean_prompt_instances': 500, 'total_distracted_prompt_instances': 3500, 'total_prompt_instances': 4000}


## Sanity check on totals

In [9]:
assert TOTAL_BASE_EXAMPLES == 250
assert TOTAL_CLEAN_PROMPT_INSTANCES == 500
assert TOTAL_DISTRACTED_PROMPT_INSTANCES == 3500
assert TOTAL_PROMPT_INSTANCES == 4000

print("Dataset-size accounting checks passed.")

Dataset-size accounting checks passed.


## Scoring notes

In [10]:
scoring_notes = [
    "All five tasks are designed to support automatic scoring.",
    "JSON outputs should be canonicalized before exact-match comparison.",
    "For multi-label classification, labels should be deduplicated and sorted before canonical comparison.",
    "For extractive QA, benchmark construction should ensure the gold answer span appears uniquely in the passage.",
    "Prompt robustness analysis should compare clean versus distracted performance and also analyze failure types by distraction family and subtype.",
]

print(json.dumps(scoring_notes, indent=2, ensure_ascii=False))

[
  "All five tasks are designed to support automatic scoring.",
  "JSON outputs should be canonicalized before exact-match comparison.",
  "For multi-label classification, labels should be deduplicated and sorted before canonical comparison.",
  "For extractive QA, benchmark construction should ensure the gold answer span appears uniquely in the passage.",
  "Prompt robustness analysis should compare clean versus distracted performance and also analyze failure types by distraction family and subtype."
]


## Build benchmark specification object

In [11]:
benchmark_spec = BenchmarkSpec(
    benchmark_name="Instruction Distraction Robustness Benchmark",
    benchmark_goal=(
        "Measure how robust instruction-following models are to common prompt distractions "
        "by comparing performance on clean and distracted prompts across bounded and unbounded regimes."
    ),
    tasks=tasks,
    distraction_families=distraction_families,
    prompt_regimes=prompt_regimes,
    bounded_regime_note=bounded_regime_note,
    dataset_size=dataset_size,
    scoring_notes=scoring_notes,
)

benchmark_spec

BenchmarkSpec(benchmark_name='Instruction Distraction Robustness Benchmark', benchmark_goal='Measure how robust instruction-following models are to common prompt distractions by comparing performance on clean and distracted prompts across bounded and unbounded regimes.', tasks=[TaskSpec(name='single_label_classification', description='Given an input text, assign exactly one label from a fixed closed label set.', output_format={'type': 'json_object', 'schema': {'label': 'string'}, 'example': {'label': 'positive'}}, scoring_rule='exact_match_on_canonical_json', constraints=['Output must be valid JSON.', "Output must contain exactly one key: 'label'.", 'The label must belong to the instance-specific allowed label set.', 'No extra keys are allowed.']), TaskSpec(name='multi_label_classification', description='Given an input text, assign all applicable labels from a fixed closed label set.', output_format={'type': 'json_object', 'schema': {'labels': ['string']}, 'example': {'labels': ['healt

## Human-readable one-page benchmark summary

In [12]:
def build_one_page_spec(spec: BenchmarkSpec) -> str:
    lines = []

    lines.append(f"Benchmark: {spec.benchmark_name}")
    lines.append("")
    lines.append("Goal:")
    lines.append(spec.benchmark_goal)
    lines.append("")
    lines.append("Prompt regimes:")
    for regime in spec.prompt_regimes:
        lines.append(f"- {regime.name}: {regime.description}")
    lines.append("")
    lines.append("Tasks:")
    for task in spec.tasks:
        lines.append(f"- {task.name}: {task.description}")
        lines.append(f"  Scoring: {task.scoring_rule}")
    lines.append("")
    lines.append("Distraction families:")
    for distraction in spec.distraction_families:
        lines.append(f"- {distraction.name}: {distraction.description}")
    lines.append("")
    lines.append("Dataset size:")
    lines.append(f"- Base examples per task: {spec.dataset_size.base_examples_per_task}")
    lines.append(f"- Number of tasks: {spec.dataset_size.num_tasks}")
    lines.append(f"- Number of distraction families: {spec.dataset_size.num_distraction_families}")
    lines.append(f"- Number of prompt regimes: {spec.dataset_size.num_regimes}")
    lines.append(f"- Total base examples: {spec.dataset_size.total_base_examples}")
    lines.append(f"- Total clean prompt instances: {spec.dataset_size.total_clean_prompt_instances}")
    lines.append(f"- Total distracted prompt instances: {spec.dataset_size.total_distracted_prompt_instances}")
    lines.append(f"- Total prompt instances: {spec.dataset_size.total_prompt_instances}")

    return "\n".join(lines)

In [13]:
one_page_spec = build_one_page_spec(benchmark_spec)
print(one_page_spec)

Benchmark: Instruction Distraction Robustness Benchmark

Goal:
Measure how robust instruction-following models are to common prompt distractions by comparing performance on clean and distracted prompts across bounded and unbounded regimes.

Prompt regimes:
- bounded: Controlled condition with explicit tagged sections. The prompt preserves visible task/input boundaries while still varying in opener, layout, and surrounding phrasing.
- unbounded: Naturalistic condition without formal tagged task/input boundaries. Prompts are rendered as message-like, memo-like, paste-like, chat-like, email-like, or workflow-like user-facing surfaces.

Tasks:
- single_label_classification: Given an input text, assign exactly one label from a fixed closed label set.
  Scoring: exact_match_on_canonical_json
- multi_label_classification: Given an input text, assign all applicable labels from a fixed closed label set.
  Scoring: exact_match_on_canonical_json_after_label_sorting
- information_extraction: Extra

## Canonicalization utilities

These are benchmark-level scoring helpers, especially for exact-match style evaluation.

In [14]:
def canonicalize_json(obj: Any) -> str:
    return json.dumps(obj, ensure_ascii=False, sort_keys=True, separators=(",", ":"))


def canonicalize_multi_label_output(obj: Dict[str, Any]) -> Dict[str, Any]:
    if "labels" not in obj or not isinstance(obj["labels"], list):
        return obj

    unique_sorted_labels = sorted(set(obj["labels"]))
    return {"labels": unique_sorted_labels}

In [15]:
example_single = {"label": "positive"}
example_multi_a = {"labels": ["tech", "health", "tech"]}
example_multi_b = {"labels": ["health", "tech"]}

print("Single label canonicalized:")
print(canonicalize_json(example_single))
print()

print("Multi-label canonicalized:")
print(canonicalize_json(canonicalize_multi_label_output(example_multi_a)))
print(canonicalize_json(canonicalize_multi_label_output(example_multi_b)))

Single label canonicalized:
{"label":"positive"}

Multi-label canonicalized:
{"labels":["health","tech"]}
{"labels":["health","tech"]}


## Exportable JSON spec

In [16]:
benchmark_spec_json = asdict(benchmark_spec)
print(json.dumps(benchmark_spec_json, indent=2, ensure_ascii=False)[:6000])

{
  "benchmark_name": "Instruction Distraction Robustness Benchmark",
  "benchmark_goal": "Measure how robust instruction-following models are to common prompt distractions by comparing performance on clean and distracted prompts across bounded and unbounded regimes.",
  "tasks": [
    {
      "name": "single_label_classification",
      "description": "Given an input text, assign exactly one label from a fixed closed label set.",
      "output_format": {
        "type": "json_object",
        "schema": {
          "label": "string"
        },
        "example": {
          "label": "positive"
        }
      },
      "scoring_rule": "exact_match_on_canonical_json",
      "constraints": [
        "Output must be valid JSON.",
        "Output must contain exactly one key: 'label'.",
        "The label must belong to the instance-specific allowed label set.",
        "No extra keys are allowed."
      ]
    },
    {
      "name": "multi_label_classification",
      "description": "Given 

## Save benchmark spec to disk

In [17]:
SPEC_OUTPUT_PATH = PROJECT_ROOT / "data" / "specs" / "benchmark_spec.json"
SPEC_OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

with open(SPEC_OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(benchmark_spec_json, f, indent=2, ensure_ascii=False)

print("Saved:", SPEC_OUTPUT_PATH)

Saved: C:\code\testing\LLMs_Robustness_Under_Distractions\data\specs\benchmark_spec.json


## Final design sanity checks

In [18]:
assert len(tasks) == 5, "Expected 5 tasks"
assert len(distraction_families) == 7, "Expected 7 distraction families"
assert len(prompt_regimes) == 2, "Expected 2 prompt regimes"
assert dataset_size.total_base_examples == 250, "Expected 250 base examples"
assert dataset_size.total_prompt_instances == 4000, "Expected 4000 total prompt instances"
assert SPEC_OUTPUT_PATH.exists(), "Benchmark spec file was not written"

print("All lock-design sanity checks passed.")

All lock-design sanity checks passed.


## Compact design summary

In [19]:
design_summary = {
    "benchmark_name": benchmark_spec.benchmark_name,
    "num_tasks": len(tasks),
    "num_distraction_families": len(distraction_families),
    "num_prompt_regimes": len(prompt_regimes),
    "total_base_examples": dataset_size.total_base_examples,
    "total_clean_prompt_instances": dataset_size.total_clean_prompt_instances,
    "total_distracted_prompt_instances": dataset_size.total_distracted_prompt_instances,
    "total_prompt_instances": dataset_size.total_prompt_instances,
}

print(json.dumps(design_summary, indent=2, ensure_ascii=False))

{
  "benchmark_name": "Instruction Distraction Robustness Benchmark",
  "num_tasks": 5,
  "num_distraction_families": 7,
  "num_prompt_regimes": 2,
  "total_base_examples": 250,
  "total_clean_prompt_instances": 500,
  "total_distracted_prompt_instances": 3500,
  "total_prompt_instances": 4000
}
